# Chess Engine with TensorFlow

## Dataset

In [1]:
import os

files = [file for file in os.listdir("../../data/pgn") if file.endswith(".pgn")]

In [2]:
len(files)

79

In [3]:
from chess import pgn

def load_pgn(file_path):
    games = []
    with open(file_path, 'r') as pgn_file:
        while True:
            game = pgn.read_game(pgn_file)
            if game is None:
                break
            games.append(game)
    return games

In [4]:
from tqdm import tqdm

LIMIT_OF_FILES = min(len(files), 15)
games = []
i = 1
for file in tqdm(files):
    games.extend(load_pgn(f"../../data/pgn/{file}"))
    if (i >= LIMIT_OF_FILES):
        break
    i += 1

 18%|█▊        | 14/79 [00:05<00:23,  2.78it/s]


In [5]:
len(games)

2582

## Build & train a neural network

In [ ]:
import numpy as np
from chess import Board
from tensorflow.keras.utils import to_categorical
from tensorflow.keras.models import Sequential
from tensorflow.keras.layers import Conv2D, Flatten, Dense
from tensorflow.keras.optimizers import Adam 
import time

In [7]:
def board_to_matrix(board: Board):
    matrix = np.zeros((8, 8, 12))
    piece_map = board.piece_map()
    for square, piece in piece_map.items():
        row, col = divmod(square, 8)
        piece_type = piece.piece_type - 1
        piece_color = 0 if piece.color else 6
        matrix[row, col, piece_type + piece_color] = 1
    return matrix


def create_input_for_nn(games):
    X = []
    y = []
    for game in games:
        board = game.board()
        for move in game.mainline_moves():
            X.append(board_to_matrix(board))
            y.append(move.uci())
            board.push(move)
    return X, y


def encode_moves(moves):
    move_to_int = {move: idx for idx, move in enumerate(set(moves))}
    return [move_to_int[move] for move in moves], move_to_int

In [8]:
X, y = create_input_for_nn(games)
y, move_to_int = encode_moves(y)
y = to_categorical(y, num_classes=len(move_to_int))
X = np.array(X)

In [9]:
model = Sequential([
    Conv2D(64, (3, 3), activation='relu', input_shape=(8, 8, 12)),
    Conv2D(128, (3, 3), activation='relu'),
    Flatten(),
    Dense(256, activation='relu'),
    Dense(len(move_to_int), activation='softmax')
])
model.compile(optimizer=Adam(), loss='categorical_crossentropy', metrics=['accuracy'])
model.summary()
model.fit(X, y, epochs=30, validation_split=0.1, batch_size=64)
model.save("../..models/TF_30EPOCHS.keras")

d:\dl project\chess-engine-main\.venv\Lib\site-packages\keras\src\layers\convolutional\base_conv.py:113: UserWarning: Do not pass an `input_shape`/`input_dim` argument to a layer. When using Sequential models, prefer using an `Input(shape)` object as the first layer in the model instead.
  super().__init__(activity_regularizer=activity_regularizer, **kwargs)


Model: "sequential"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv2d (Conv2D)                 │ (None, 6, 6, 64)       │         6,976 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv2d_1 (Conv2D)               │ (None, 4, 4, 128)      │        73,856 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ flatten (Flatten)               │ (None, 2048)           │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense (Dense)                   │ (None, 256)            │       524,544 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_1 (Dense)                 │ (None, 1831)           │       470,567 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 1,075,943 (4.10 MB)

 Trainable params: 1,075,943 (4.10 MB)

 Non-trainable params: 0 (0.00 B)

Epoch 1/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 24s 7ms/step - accuracy: 0.0806 - loss: 5.8515 - val_accuracy: 0.1206 - val_loss: 5.3230
Epoch 2/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 22s 7ms/step - accuracy: 0.1397 - loss: 4.8942 - val_accuracy: 0.1403 - val_loss: 4.8419
Epoch 3/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 22s 7ms/step - accuracy: 0.1694 - loss: 4.3273 - val_accuracy: 0.1508 - val_loss: 4.6937
Epoch 4/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 22s 7ms/step - accuracy: 0.1962 - loss: 3.9601 - val_accuracy: 0.1571 - val_loss: 4.6720
Epoch 5/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - accuracy: 0.2195 - loss: 3.6917 - val_accuracy: 0.1571 - val_loss: 4.7267
Epoch 6/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 21s 7ms/step - accuracy: 0.2429 - loss: 3.4747 - val_accuracy: 0.1583 - val_loss: 4.8063
Epoch 7/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 37s 12ms/step - accuracy: 0.2650 - loss: 3.2917 - val_accuracy: 0.1541 - val_loss: 4.9055
Epoch 8/30
3040/3040 ━━━━━━━━━━━━━━━━━━━━ 46s 15ms/step - accuracy: 0.2852 - loss:

FileNotFoundError: [Errno 2] No such file or directory: 'models/TF_30EPOCHS.keras'

In [12]:
import os

os.makedirs("../../models", exist_ok=True)
model.save("../../models/TF_30EPOCHS.keras")

## Predictions

In [2]:

from tensorflow.keras.models import load_model
model = load_model("../../models/TF_30EPOCHS.keras")

In [4]:
int_to_move = dict(zip(move_to_int.values(), move_to_int.keys()))


def predict_next_move(board):
    board_matrix = board_to_matrix(board).reshape(1, 8, 8, 12)
    predictions = model.predict(board_matrix)[0]
    legal_moves = list(board.legal_moves)
    legal_moves_uci = [move.uci() for move in legal_moves]
    sorted_indices = np.argsort(predictions)[::-1]
    for move_index in sorted_indices:
        move = int_to_move[move_index]
        if move in legal_moves_uci:
            return move
    return None

NameError: name 'move_to_int' is not defined

In [5]:
import random
import math
import chess
import numpy as np

try:
    import ipywidgets as widgets
    from IPython.display import clear_output, display
    HAS_WIDGETS = True
except Exception:
    HAS_WIDGETS = False
    widgets = None


PIECE_VALUES = {
    chess.PAWN: 1,
    chess.KNIGHT: 3,
    chess.BISHOP: 3,
    chess.ROOK: 5,
    chess.QUEEN: 9,
    chess.KING: 0,
}

GAME_STATE = {
    "board": None,
    "human_is_white": True,
    "active": False,
}

GAME_UI = {
    "output": None,
    "selected_from": None,
    "flipped": False,
    "enabled": False,
    "ai_mode": "hybrid",
    "last_move": None,
    "status": "",
   
    "ai_temperature": 0.95,
    "ai_top_k": 6,
    "ai_epsilon": 0.15,
    "novelty_weight": 0.25,
    "position_counts": {},
}


def _position_key(board: chess.Board):
    return f"{board.board_fen()} {'w' if board.turn == chess.WHITE else 'b'}"


def _register_position(board: chess.Board):
    key = _position_key(board)
    counts = GAME_UI.get("position_counts", {})
    counts[key] = counts.get(key, 0) + 1
    GAME_UI["position_counts"] = counts


def _material_score(board: chess.Board, bot_color: bool) -> float:
    score = 0.0
    for _, piece in board.piece_map().items():
        value = PIECE_VALUES[piece.piece_type]
        score += value if piece.color == bot_color else -value
    return score


def _model_prior_for_moves(board: chess.Board, legal_moves):
    priors = {}
    if not ("model" in globals() and "int_to_move" in globals() and "board_to_matrix" in globals()):
        return priors

    try:
        board_matrix = board_to_matrix(board).reshape(1, 8, 8, 12)
        predictions = model.predict(board_matrix, verbose=0)[0]
        move_to_score = {}
        for idx, uci in int_to_move.items():
            if idx < len(predictions):
                move_to_score[uci] = float(predictions[idx])
        for mv in legal_moves:
            priors[mv] = move_to_score.get(mv.uci(), 0.0)
    except Exception:
        return {}

    return priors


def _sample_from_candidates(candidates, temperature: float):
    """candidates: list[(move, score)]"""
    if len(candidates) == 1:
        return candidates[0][0]

    temp = max(0.05, float(temperature))
    scores = np.array([s for _, s in candidates], dtype=float)
    scores = scores - np.max(scores)
    logits = scores / temp
    logits = np.clip(logits, -50, 50)
    probs = np.exp(logits)
    probs_sum = probs.sum()
    if probs_sum <= 0 or not np.isfinite(probs_sum):
        return random.choice([m for m, _ in candidates])
    probs = probs / probs_sum
    idx = np.random.choice(len(candidates), p=probs)
    return candidates[idx][0]


def choose_ai_move(board: chess.Board):
    """Non-deterministic AI selector with anti-repeat behavior.

    Modes:
    - fast   : tactical/material one-ply
    - hybrid : fast + model prior (default)
    - model  : model-heavy scoring
    """
    legal_moves = list(board.legal_moves)
    if not legal_moves:
        return None

  
    epsilon = float(GAME_UI.get("ai_epsilon", 0.15))
    if random.random() < epsilon:
        return random.choice(legal_moves)

    bot_color = board.turn
    mode = GAME_UI.get("ai_mode", "hybrid")
    priors = _model_prior_for_moves(board, legal_moves) if mode in {"hybrid", "model"} else {}
    novelty_weight = float(GAME_UI.get("novelty_weight", 0.25))

    scored = []
    for move in legal_moves:
        capture_piece = board.piece_at(move.to_square)
        board.push(move)

        if board.is_checkmate():
            score = 10_000.0
        else:
            material = _material_score(board, bot_color)
            check_bonus = 0.35 if board.is_check() else 0.0
            capture_bonus = PIECE_VALUES[capture_piece.piece_type] * 0.7 if capture_piece else 0.0

            center_bonus = 0.15 if move.to_square in {chess.D4, chess.E4, chess.D5, chess.E5} else 0.0

            model_bonus = 0.0
            if move in priors:
                model_bonus = priors[move] * (0.9 if mode == "model" else 0.35)

       
            next_key = _position_key(board)
            seen_count = GAME_UI.get("position_counts", {}).get(next_key, 0)
            novelty_penalty = seen_count * novelty_weight

     
            opening_noise = 0.0
            if len(board.move_stack) <= 14:
                opening_noise = random.uniform(-0.12, 0.12)

            score = (material * 1.2) + check_bonus + capture_bonus + center_bonus + model_bonus - novelty_penalty + opening_noise

        board.pop()
        scored.append((move, score))

    scored.sort(key=lambda x: x[1], reverse=True)
    top_k = max(1, int(GAME_UI.get("ai_top_k", 6)))
    candidates = scored[: min(top_k, len(scored))]

    temperature = float(GAME_UI.get("ai_temperature", 0.95))
    return _sample_from_candidates(candidates, temperature)


def set_bot_variety(level: str = "medium"):
    """Quick presets: low, medium, high"""
    level = level.lower().strip()
    if level == "low":
        GAME_UI["ai_temperature"] = 0.65
        GAME_UI["ai_top_k"] = 3
        GAME_UI["ai_epsilon"] = 0.05
        GAME_UI["novelty_weight"] = 0.12
    elif level == "high":
        GAME_UI["ai_temperature"] = 1.2
        GAME_UI["ai_top_k"] = 10
        GAME_UI["ai_epsilon"] = 0.24
        GAME_UI["novelty_weight"] = 0.35
    else: 
        GAME_UI["ai_temperature"] = 0.95
        GAME_UI["ai_top_k"] = 6
        GAME_UI["ai_epsilon"] = 0.15
        GAME_UI["novelty_weight"] = 0.25


def parse_human_move(board: chess.Board, user_text: str):
    user_text = user_text.strip()
    if not user_text:
        return None

    try:
        return board.parse_san(user_text)
    except Exception:
        pass

    try:
        move = chess.Move.from_uci(user_text)
        if move in board.legal_moves:
            return move
    except Exception:
        pass

    return None


def _print_result(board: chess.Board):
    print("Result:", board.result())
    if board.is_checkmate():
        print("Checkmate.")
    elif board.is_stalemate():
        print("Stalemate.")
    elif board.is_insufficient_material():
        print("Draw (insufficient material).")


def _can_select_square(board: chess.Board, square: chess.Square):
    piece = board.piece_at(square)
    if piece is None:
        return False
    human_is_white = GAME_STATE["human_is_white"]
    return piece.color == (chess.WHITE if human_is_white else chess.BLACK)


def _rank_file_lists(flipped: bool):
    ranks = list(range(8, 0, -1))
    files = list("abcdefgh")
    if flipped:
        ranks = list(range(1, 9))
        files = list("hgfedcba")
    return ranks, files


def _legal_targets_from_selected(board: chess.Board):
    selected = GAME_UI.get("selected_from")
    if selected is None:
        return set()
    targets = set()
    for mv in board.legal_moves:
        if mv.from_square == selected:
            targets.add(mv.to_square)
    return targets


def _square_display(board: chess.Board, square: chess.Square):
    piece = board.piece_at(square)
    return piece.unicode_symbol() if piece else " "


def _square_color(square: chess.Square):
    
    is_dark = (chess.square_file(square) + chess.square_rank(square)) % 2 == 0
    return "#b58863" if is_dark else "#f0d9b5"


def _build_click_board(board: chess.Board, on_square_click, flipped: bool):
    ranks, files = _rank_file_lists(flipped)
    selected = GAME_UI.get("selected_from")
    legal_targets = _legal_targets_from_selected(board)
    last_move = GAME_UI.get("last_move")

    top_labels = widgets.HBox([
        widgets.HTML(value=f"<b style='margin:0 18px;font-size:16px'>{f}</b>") for f in files
    ])

    rows = []
    for rank in ranks:
        row_buttons = []
        for file_char in files:
            square = chess.parse_square(f"{file_char}{rank}")
            btn = widgets.Button(
                description=_square_display(board, square),
                layout=widgets.Layout(width="68px", height="68px", padding="0"),
            )
            btn.style.font_size = "34px"

            color = _square_color(square)

            if last_move is not None and (square == last_move.from_square or square == last_move.to_square):
                color = "#7fb3ff"
            if square in legal_targets:
                color = "#f6f08f"
            if selected == square:
                color = "#7ec97e"

            if board.is_check():
                king_square = board.king(board.turn)
                if king_square == square:
                    color = "#ff8a80"

            btn.style.button_color = color

            def _handler(b, sq=square):
                on_square_click(sq)

            btn.on_click(_handler)
            row_buttons.append(btn)

        row = widgets.HBox(
            [widgets.HTML(value=f"<b style='width:20px;display:inline-block;font-size:16px'>{rank}</b>")] + row_buttons + [widgets.HTML(value=f"<b style='width:20px;display:inline-block;font-size:16px'>{rank}</b>")],
            layout=widgets.Layout(margin="0"),
        )
        rows.append(row)

    bottom_labels = widgets.HBox([
        widgets.HTML(value=f"<b style='margin:0 18px;font-size:16px'>{f}</b>") for f in files
    ])

    return widgets.VBox(
        [top_labels] + rows + [bottom_labels],
        layout=widgets.Layout(border="2px solid #333", width="fit-content", padding="8px"),
    )


def _recent_moves_text(board: chess.Board, max_pairs: int = 6):
    if not board.move_stack:
        return "No moves yet."
    temp = chess.Board()
    san_moves = []
    for mv in board.move_stack:
        san_moves.append(temp.san(mv))
        temp.push(mv)

    pairs = []
    for i in range(0, len(san_moves), 2):
        move_no = i // 2 + 1
        if i + 1 < len(san_moves):
            pairs.append(f"{move_no}. {san_moves[i]} {san_moves[i + 1]}")
        else:
            pairs.append(f"{move_no}. {san_moves[i]}")
    return " | ".join(pairs[-max_pairs:])


def _try_human_click_move(target_square: chess.Square):
    board = GAME_STATE["board"]
    selected_from = GAME_UI.get("selected_from")

    if selected_from is None:
        if _can_select_square(board, target_square):
            GAME_UI["selected_from"] = target_square
            return "Piece selected. Click a highlighted destination."
        return "Select one of your pieces first."

    move = chess.Move(selected_from, target_square)
    piece = board.piece_at(selected_from)

    if piece and piece.piece_type == chess.PAWN:
        target_rank = chess.square_rank(target_square)
        if target_rank in (0, 7):
            move = chess.Move(selected_from, target_square, promotion=chess.QUEEN)

    GAME_UI["selected_from"] = None

    if move not in board.legal_moves:
        return "Illegal move. Try another destination."

    board.push(move)
    GAME_UI["last_move"] = move
    _register_position(board)

    if board.is_game_over():
        GAME_STATE["active"] = False
        return f"You played {move.uci()}. Game over: {board.result()}"

    ai_move = choose_ai_move(board)
    if ai_move is None:
        GAME_STATE["active"] = False
        return f"You played {move.uci()}. AI has no legal moves."

    board.push(ai_move)
    GAME_UI["last_move"] = ai_move
    _register_position(board)

    if board.is_game_over():
        GAME_STATE["active"] = False
        return f"You played {move.uci()}. AI played {ai_move.uci()}. Game over: {board.result()}"

    return f"You played {move.uci()}. AI played {ai_move.uci()}."


def render_click_board(message: str = ""):
    if not HAS_WIDGETS:
        print("ipywidgets is not installed.")
        return None

    board = GAME_STATE["board"]
    output = GAME_UI.get("output")
    if board is None or output is None:
        print("Start with launch_click_to_move_ui('white' or 'black').")
        return None

    with output:
        clear_output(wait=True)
        display(_build_click_board(board, _handle_square_click, GAME_UI["flipped"]))

        if message:
            print(message)
        print(f"Turn: {'White' if board.turn == chess.WHITE else 'Black'}")
        print(f"Mode: {GAME_UI.get('ai_mode', 'hybrid').upper()} | Variety: temp={GAME_UI.get('ai_temperature')} top_k={GAME_UI.get('ai_top_k')} eps={GAME_UI.get('ai_epsilon')}")
        print(f"Moves: {_recent_moves_text(board)}")

        if board.is_game_over():
            _print_result(board)

    return board


def _handle_square_click(square: chess.Square):
    board = GAME_STATE["board"]
    if board is None or not GAME_STATE["active"]:
        render_click_board("No active game. Start again.")
        return

    human_turn = (board.turn == chess.WHITE and GAME_STATE["human_is_white"]) or (
        board.turn == chess.BLACK and not GAME_STATE["human_is_white"]
    )

    if not human_turn:
        render_click_board("Wait for AI turn.")
        return

    msg = _try_human_click_move(square)
    render_click_board(msg)


def launch_click_to_move_ui(human_color: str = "white", ai_mode: str = "hybrid", variety: str = "medium"):
    """Launch one-board clickable UI.

    ai_mode: 'fast', 'hybrid', or 'model'
    variety: 'low', 'medium', or 'high'
    """
    if not HAS_WIDGETS:
        print("ipywidgets not available.")
        return None

    human_color = human_color.lower().strip()
    ai_mode = ai_mode.lower().strip()
    variety = variety.lower().strip()

    if human_color not in {"white", "black"}:
        raise ValueError("human_color must be 'white' or 'black'")
    if ai_mode not in {"fast", "hybrid", "model"}:
        raise ValueError("ai_mode must be 'fast', 'hybrid', or 'model'")
    if variety not in {"low", "medium", "high"}:
        raise ValueError("variety must be 'low', 'medium', or 'high'")

    GAME_STATE["board"] = chess.Board()
    GAME_STATE["human_is_white"] = human_color == "white"
    GAME_STATE["active"] = True

    GAME_UI["selected_from"] = None
    GAME_UI["flipped"] = human_color == "black"
    GAME_UI["enabled"] = True
    GAME_UI["ai_mode"] = ai_mode
    GAME_UI["last_move"] = None
    GAME_UI["position_counts"] = {}
    GAME_UI["output"] = widgets.Output()

    set_bot_variety(variety)
    _register_position(GAME_STATE["board"])

    display(GAME_UI["output"])
    intro = f"Single-board mode. You are {human_color.upper()} | AI mode: {ai_mode.upper()} | Variety: {variety.upper()}"

    if human_color == "black":
        ai_move = choose_ai_move(GAME_STATE["board"])
        if ai_move is not None:
            GAME_STATE["board"].push(ai_move)
            GAME_UI["last_move"] = ai_move
            _register_position(GAME_STATE["board"])
            intro += f" | AI opened with {ai_move.uci()}"

    return render_click_board(intro)

In [6]:

_ = launch_click_to_move_ui("white", ai_mode="hybrid", variety="high")

Output()